#### DUPLICITY PROFILE


##### 06.1 DOCUMENTAÇÃO DO DUPLICITY PROFILE

###### Objetivo

O **Duplicity Profile** tem como objetivo identificar, quantificar e analisar a ocorrência de duplicidades nos registros e atributos do DataFrame.

Enquanto o **Schema Profile** analisa a estrutura das colunas, o **Distribution Profile** analisa a distribuição dos valores e o **Pattern Profile** identifica padrões estruturais, o **Duplicity Profile** avalia a repetição de informações e o potencial de unicidade dos dados.

A análise não depende do significado de negócio das colunas.

O foco está nas características observáveis dos dados, como:

- duplicidade completa de registros;
- grupos de registros idênticos;
- quantidade de registros envolvidos em duplicações;
- duplicações excedentes;
- frequência de repetição;
- unicidade por coluna;
- valores distintos e repetidos;
- possíveis chaves candidatas;
- possíveis chaves compostas;
- concentração dos grupos duplicados.

###### Exemplos

Considere os registros:

| ID | Produto | Quantidade |
|---|---|---|
| 101 | Produto A | 2 |
| 101 | Produto A | 2 |
| 102 | Produto B | 3 |
| 103 | Produto C | 1 |
| 103 | Produto C | 1 |
| 103 | Produto C | 1 |

Nesse conjunto existem:

- 6 registros no total;
- 3 combinações distintas de registros;
- 2 grupos com duplicidade;
- 5 registros envolvidos em grupos duplicados;
- 3 registros duplicados excedentes.

A duplicidade excedente corresponde à quantidade de registros que ultrapassa uma ocorrência por grupo.

Neste exemplo:

`Duplicações excedentes = 6 - 3 = 3`

O percentual de duplicidade excedente é:

`(3 / 6) × 100 = 50%`

Já o percentual de registros envolvidos em grupos duplicados é:

`(5 / 6) × 100 = 83,33%`

Esses indicadores representam perspectivas diferentes da duplicidade.

###### Principais análises

- Identificação de registros completamente duplicados;
- Contagem de combinações distintas de registros;
- Quantidade de grupos com duplicidade;
- Quantidade de registros envolvidos em duplicações;
- Quantidade e percentual de duplicações excedentes;
- Frequência dos grupos duplicados;
- Ranking dos grupos com maior repetição;
- Quantidade de valores distintos por coluna;
- Percentual de unicidade por coluna;
- Identificação de possíveis chaves candidatas simples;
- Avaliação controlada de possíveis chaves compostas;
- Consolidação dos indicadores de duplicidade;
- Resumo executivo do Duplicity Profile.

###### Interpretação dos resultados

A ocorrência de valores repetidos não representa necessariamente um problema de qualidade dos dados.

Uma coluna que representa município, categoria ou situação operacional pode apresentar repetições legítimas.

Da mesma forma, registros completamente idênticos podem ocorrer em determinadas fontes sem representar necessariamente um erro.

Por esse motivo, o **Duplicity Profile** diferencia:

- **Repetição de valores:** ocorrência de um mesmo valor em uma coluna;
- **Duplicidade completa:** ocorrência de registros idênticos considerando todas as colunas analisadas;
- **Registros envolvidos em duplicações:** total de registros pertencentes a grupos com mais de uma ocorrência;
- **Duplicações excedentes:** quantidade de ocorrências adicionais à primeira de cada grupo;
- **Unicidade:** capacidade de uma coluna ou combinação de colunas apresentar valores exclusivos;
- **Chave candidata:** atributo ou combinação de atributos com potencial para identificar registros de forma única.

As chaves identificadas são consideradas candidatas estatísticas e não são automaticamente classificadas como chaves de negócio.

A confirmação de uma regra de unicidade depende do contexto e dos requisitos da fonte de dados.

###### Característica

O **Duplicity Profile** é independente da origem dos dados.

Sua análise é realizada exclusivamente sobre o DataFrame disponibilizado pelo **Data Preparation**.

O processamento utiliza operações distribuídas do Apache Spark para analisar registros, valores distintos e combinações de atributos.

A avaliação de chaves compostas utiliza limites configuráveis para evitar a exploração indiscriminada de combinações de colunas e reduzir o custo computacional.

As funções genéricas e reutilizáveis são mantidas no notebook de bibliotecas **00_FRAMEWORK_CONFIG**, enquanto o **Duplicity Profile** concentra as configurações específicas, análises, consolidações e visualizações.

Dessa forma, o mesmo Profile pode ser utilizado em diferentes fontes, volumes e estruturas de dados.

###### Resultado esperado

Ao final desta etapa estarão disponíveis:

- total de registros analisados;
- quantidade de registros distintos;
- quantidade de grupos duplicados;
- quantidade de registros envolvidos em duplicações;
- quantidade e percentual de duplicações excedentes;
- ranking dos grupos duplicados mais frequentes;
- indicadores de unicidade por coluna;
- identificação de possíveis chaves candidatas simples;
- identificação de possíveis chaves compostas;
- métricas consolidadas de duplicidade;
- resumo executivo dos resultados;
- DataFrame consolidado do Duplicity Profile.

Os principais DataFrames disponibilizados são:

- `duplicity_profile_df`
- `duplicity_summary_df`
- `duplicity_full_groups_df`
- `duplicity_top_groups_df`
- `duplicity_composite_keys_df`

Esses resultados poderão ser utilizados posteriormente pelo **Data Quality Score**, permitindo incorporar evidências de duplicidade e unicidade à avaliação geral da qualidade dos dados, respeitando a aplicabilidade de cada métrica.


In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 06.3 CONFIGURAÇÕES
# ============================================================

# O QUE FAZ: 
# Define limites e política de valores.

# COMO FAZ: 
# Utiliza parâmetros locais do Profile.

# POR QUE É IMPORTANTE: 
# Permite controlar custo e interpretação.

# PERGUNTA RESPONDIDA: 
# "Quais regras e limites serão aplicados?"

from itertools import combinations

DUPLICITY_TOP_N = 5
DUPLICITY_MAX_CANDIDATE_COLUMNS = 8
DUPLICITY_MAX_COMPOSITE_PAIRS = 15
DUPLICITY_ENABLE_COMPOSITE = True
DUPLICITY_IGNORE_BLANK_STRINGS = True
DUPLICITY_SHOW_RESULTS = True

# Pares são testados apenas quando nenhuma das duas colunas já é chave simples.
# A seleção é heurística: não garante encontrar todas as chaves compostas.

In [0]:
# ============================================================
# 06.4 VALIDAÇÃO E ELEGIBILIDADE
# ============================================================

# O QUE FAZ: 
# Verifica entrada e suporte de tipos.

# COMO FAZ:
#  Usa o schema e total já preparados.

# POR QUE É IMPORTANTE: 
# Evita execução silenciosa com dados inadequados.

# PERGUNTA RESPONDIDA: 
# "Quais colunas podem ser analisadas?"

if "df_prepared" not in locals() or "total_registros" not in locals():
    raise ValueError("Execute 02_DATA_PREPARATION antes do Duplicity Profile.")
if total_registros <= 0 or not df_prepared.columns:
    raise ValueError("O Duplicity Profile requer registros e colunas.")
if len(set(df_prepared.columns)) != len(df_prepared.columns):
    raise ValueError("Nomes de colunas duplicados não são suportados.")
for nome in ("dq_col", "dq_valid_value", "dq_is_atomic", "dq_duplicate_group_metrics"):
    if nome not in globals():
        raise RuntimeError(f"Função {nome} não encontrada. Atualize 00_FRAMEWORK_CONFIG.")

campos = df_prepared.schema.fields
tipos = {campo.name: campo.dataType for campo in campos}
atomicas = [f.name for f in campos if dq_is_atomic(f.dataType)]
complexas = [f.name for f in campos if not dq_is_atomic(f.dataType)]
print(f"Registros: {total_registros:,} | Colunas: {len(campos)}")
print(f"Colunas não elegíveis: {complexas}")

In [0]:
# ============================================================
# 06.5 DUPLICIDADE COMPLETA
# ============================================================

# O QUE FAZ: 
# Quantifica grupos de linhas idênticas.

# COMO FAZ: 
# Agrupa todas as colunas e consolida frequências.

# POR QUE É IMPORTANTE: 
# Distingue excedentes de envolvidos.

# PERGUNTA RESPONDIDA: 
# "Quantas linhas completas estão duplicadas?"

duplicity_full_status = "EXECUTADO" if not complexas else "NAO_EXECUTADO_TIPOS_COMPLEXOS"
duplicity_full_groups_df = None
duplicity_top_groups_df = None
if duplicity_full_status == "EXECUTADO":
    duplicity_full_groups_df = df_prepared.groupBy(*[dq_col(c) for c in atomicas]).agg(
        F.count(F.lit(1)).alias("ocorrencias")
    )
    resumo_linhas = dq_duplicate_group_metrics(duplicity_full_groups_df).first().asDict()
    duplicity_top_groups_df = (
        duplicity_full_groups_df.filter(F.col("ocorrencias") > 1)
        .orderBy(F.desc("ocorrencias")).limit(DUPLICITY_TOP_N)
    )
else:
    resumo_linhas = {k: None for k in (
        "grupos_distintos", "grupos_duplicados", "registros_em_grupos_duplicados",
        "linhas_duplicadas_excedentes", "maior_grupo")}

# Schema explícito: funciona mesmo se a análise não puder ser executada.
full_schema = """status string, total_registros long, grupos_distintos long,
grupos_duplicados long, registros_em_grupos_duplicados long,
linhas_duplicadas_excedentes long, maior_grupo long,
pct_linhas_duplicadas_excedentes double, pct_registros_em_grupos_duplicados double"""
resumo_full = (
    duplicity_full_status, int(total_registros),
    *[int(resumo_linhas[k]) if resumo_linhas[k] is not None else None for k in (
        "grupos_distintos", "grupos_duplicados", "registros_em_grupos_duplicados",
        "linhas_duplicadas_excedentes", "maior_grupo")],
    (100.0 * resumo_linhas["linhas_duplicadas_excedentes"] / total_registros
     if resumo_linhas["linhas_duplicadas_excedentes"] is not None else None),
    (100.0 * resumo_linhas["registros_em_grupos_duplicados"] / total_registros
     if resumo_linhas["registros_em_grupos_duplicados"] is not None else None)
)
duplicity_full_summary_df = spark.createDataFrame([resumo_full], full_schema)
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_full_summary_df)
    if duplicity_top_groups_df is not None:
        display(duplicity_top_groups_df)

In [0]:
# ============================================================
# 06.6 UNICIDADE POR COLUNA
# ============================================================

# O QUE FAZ: 
# Calcula validade e unicidade de cada coluna.

# COMO FAZ: 
# Constrói agregações exatas em um único plano Spark.

# POR QUE É IMPORTANTE: 
# Identifica chaves candidatas sem presumir obrigatoriedade.

# PERGUNTA RESPONDIDA: 
# "Quais colunas são únicas e totalmente preenchidas?"

metricas = []
for i, campo in enumerate(campos):
    if not dq_is_atomic(campo.dataType):
        continue
    c = dq_col(campo.name)
    valido = dq_valid_value(campo.name, campo.dataType, DUPLICITY_IGNORE_BLANK_STRINGS)
    metricas.extend([
        F.sum(F.when(c.isNull(), 1).otherwise(0)).alias(f"m{i}_null"),
        F.sum(F.when(c.isNotNull() & (~valido), 1).otherwise(0)).alias(f"m{i}_invalidos_nao_null"),
        F.sum(F.when(valido, 1).otherwise(0)).alias(f"m{i}_validos"),
        F.countDistinct(F.when(valido, c)).alias(f"m{i}_distintos")
    ])
row = df_prepared.agg(*metricas).first() if metricas else None
resultados_colunas = []
for i, campo in enumerate(campos):
    if not dq_is_atomic(campo.dataType):
        resultados_colunas.append((campo.name, campo.dataType.simpleString(),
            "NAO_ELEGIVEL_TIPO_COMPLEXO", None, None, None, None, None, None, False))
        continue
    nulos = int(row[f"m{i}_null"] or 0)
    invalidos = int(row[f"m{i}_invalidos_nao_null"] or 0)
    validos = int(row[f"m{i}_validos"] or 0)
    distintos = int(row[f"m{i}_distintos"] or 0)
    chave = validos == total_registros and distintos == total_registros
    resultados_colunas.append((campo.name, campo.dataType.simpleString(),
        "EXECUTADO", nulos, invalidos, validos, distintos, validos-distintos,
        100.0*distintos/validos if validos else None, chave))

schema_colunas = """coluna string, tipo_spark string, status string, null long,
invalidos_nao_null long, valores_validos long, distintos_validos long,
repeticoes_excedentes long, pct_unicidade_validos double, chave_candidata_simples boolean"""
duplicity_column_profile_df = spark.createDataFrame(resultados_colunas, schema_colunas)
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_column_profile_df.orderBy(F.desc("pct_unicidade_validos")))

In [0]:
# ============================================================
# 06.7 CHAVES COMPOSTAS
# ============================================================

# O QUE FAZ: 
# Testa pares candidatos com limites de custo.

# COMO FAZ: 
# Seleciona colunas elegíveis e agrega pares por struct.

# POR QUE É IMPORTANTE: 
# Evita exploração combinatória irrestrita.

# PERGUNTA RESPONDIDA: 
# "Quais pares podem identificar unicamente registros?"

# Seleção heurística; não é busca exaustiva de chaves compostas.
selecionadas = (duplicity_column_profile_df
    .filter((F.col("status") == "EXECUTADO") & (F.col("valores_validos") > 0)
            & (~F.col("chave_candidata_simples")))
    .orderBy(F.desc("pct_unicidade_validos"), F.asc("coluna"))
    .limit(DUPLICITY_MAX_CANDIDATE_COLUMNS)
    .select("coluna").collect())
nomes_candidatos = [r["coluna"] for r in selecionadas]
pares = (list(combinations(nomes_candidatos, 2))[:DUPLICITY_MAX_COMPOSITE_PAIRS]
         if DUPLICITY_ENABLE_COMPOSITE else [])
expressoes = []
for i, (a,b) in enumerate(pares):
    ca, cb = dq_col(a), dq_col(b)
    valido = (dq_valid_value(a, tipos[a], DUPLICITY_IGNORE_BLANK_STRINGS)
              & dq_valid_value(b, tipos[b], DUPLICITY_IGNORE_BLANK_STRINGS))
    expressoes.extend([
        F.sum(F.when(valido, 1).otherwise(0)).alias(f"p{i}_validos"),
        F.countDistinct(F.when(valido, F.struct(ca.alias("a"), cb.alias("b")))).alias(f"p{i}_distintos")
    ])
pair_row = df_prepared.agg(*expressoes).first() if expressoes else None
pair_results = []
for i, (a,b) in enumerate(pares):
    validos = int(pair_row[f"p{i}_validos"] or 0)
    distintos = int(pair_row[f"p{i}_distintos"] or 0)
    pair_results.append((a,b,validos,distintos,validos-distintos,
        100.0*distintos/validos if validos else None,
        validos == total_registros and distintos == total_registros))
pair_schema = """coluna_a string, coluna_b string, pares_validos long,
pares_distintos long, repeticoes_excedentes long,
pct_unicidade_pares_validos double, chave_candidata_composta boolean"""
# Schema explícito permite criar DataFrame mesmo quando nenhum par é elegível.
duplicity_composite_keys_df = spark.createDataFrame(pair_results, pair_schema)
print(f"Pares avaliados: {len(pares)}")
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_composite_keys_df.orderBy(F.desc("pct_unicidade_pares_validos")))

In [0]:
# ============================================================
# 06.8 CONSOLIDAÇÃO E RESUMO EXECUTIVO
# ============================================================

# O QUE FAZ: 
# Expõe saídas padronizadas.

# COMO FAZ: 
# Reúne indicadores de colunas, linhas e pares.

# POR QUE É IMPORTANTE: 
# Facilita consumo posterior pelo Data Quality Score.

# PERGUNTA RESPONDIDA: 
# "Quais evidências foram produzidas?"

duplicity_profile_df = duplicity_column_profile_df
qtd_chaves_simples = sum(1 for r in resultados_colunas if r[-1])
qtd_chaves_compostas = sum(1 for r in pair_results if r[-1])
duplicity_summary_df = (duplicity_full_summary_df
    .withColumn("colunas_chave_simples", F.lit(qtd_chaves_simples))
    .withColumn("pares_avaliados", F.lit(len(pares)))
    .withColumn("pares_chave_composta", F.lit(qtd_chaves_compostas))
    .withColumn("colunas_nao_elegiveis", F.lit(len(complexas))))
if DUPLICITY_SHOW_RESULTS:
    display(duplicity_summary_df)
print("Saídas: duplicity_profile_df, duplicity_summary_df, duplicity_full_groups_df,")
print("duplicity_top_groups_df, duplicity_composite_keys_df")

In [0]:
# ============================================================
# 06.9 TESTES DE SANIDADE
# ============================================================

# O QUE FAZ: 
# Valida invariantes de duplicidade e unicidade.

# COMO FAZ: 
# Usa dataset sintético de seis registros.

# POR QUE É IMPORTANTE: 
# Detecta regressões em fórmulas básicas.

# PERGUNTA RESPONDIDA: 
# "As métricas reproduzem resultados conhecidos?"

teste = spark.createDataFrame([
    ("A", 1, "x"), ("A", 1, "x"), ("A", 1, "y"),
    ("B", 2, "z"), ("B", 2, "z"), ("C", 3, "w")
], ["codigo", "valor", "rotulo"])
grupos_teste = teste.groupBy("codigo", "valor", "rotulo").count().withColumnRenamed("count", "ocorrencias")
t = dq_duplicate_group_metrics(grupos_teste).first()
assert t["grupos_distintos"] == 4
assert t["grupos_duplicados"] == 2
assert t["registros_em_grupos_duplicados"] == 4
assert t["linhas_duplicadas_excedentes"] == 2
assert t["linhas_duplicadas_excedentes"] == 6 - t["grupos_distintos"]

strings = spark.createDataFrame([("A",), ("",), ("   ",), (None,)], ["codigo"])
assert strings.filter(dq_valid_value("codigo", StringType(), True)).count() == 1
assert strings.filter(dq_valid_value("codigo", StringType(), False)).count() == 3

# Par (codigo, rotulo) não é chave: há linhas repetidas.
assert teste.select("codigo", "rotulo").distinct().count() == 4
print("Testes matemáticos sintéticos: OK")